## Uncertainty Engine SDK: Example Materials Problem

#### Import Functions

The [Uncertainty Engine SDK](https://pypi.org/project/uncertainty-engine/) follows a graph-based programming paradigm. This notebook is designed for python users more familiar with a functional programming approach. For this, we wrap the machine learning workflows (graphs) into functions used throughout this notebook. Head to `utils.py` to see what is happening under the hood.

In [ ]:
from utils import (
    simulate,
    plot_basic, 
    plot_model_uncertainty, 
    get_dataset, 
    add_new_sample, 
    train_model, 
    create_visualise_dataset, 
    recommend_new_sample
    )


import numpy as np
import pandas as pd 

#### Establish UE Connection and Authenticate

*NOTE* When authenticating your Uncertainty Engine (UE) access, you must provide your username (email) and password. This is done by setting them as environment variables. Please be careful when committing so not to accidentally share credentials. For collaborative repositories, it is recommended to use a `.env` file alongside [dotenv](https://pypi.org/project/python-dotenv/) and ensure `.env` is added to your `.gitignore`.

*NOTE* Currently, Uncertainty Engine projects must be made via the GUI. Head to [uncertaintyengine.ai](uncertaintyengine.ai) to add a new project.

In [ ]:
import os
from uncertainty_engine import Client 
from dotenv import load_dotenv

# Load the environment variables (stored in .env in root directory of project)
load_dotenv()

# # Enter your UE username and password here (if .env not set up)
# os.environ["UE_USERNAME"] = ""
# os.environ["UE_PASSWORD"] = ""

client = Client()
client.authenticate()

# A project must be created in the Uncertainty Engine GUI (uncertaintyengine.ai) manually. 
# Please change the project name here in accordance with the project name set inm the GUI.
PROJECT_NAME = "[ENTER PROJECT NAME]"
# Extract the project ID
project_id = client.projects.get_project_id_by_name(PROJECT_NAME)

### 1. Upload Initial Sample Set to UE

In [ ]:
# Create initial sample set
x_train = np.array([0, 5])
# The simulation function can be changed in utils.py to suit the particular problem
# NB: we add noise to 'perfect' simulation (reflecting ground truth)
y_train = simulate(x_train, noise=True)

# Create data_local directory, if it doesn't already exist
os.makedirs("data_local", exist_ok=True)

# Save as csvs
df = pd.DataFrame(x_train, columns=["x"])
df.to_csv("data_local/x_train_1.csv", index=False)
df = pd.DataFrame(y_train, columns=["y"])
df.to_csv("data_local/y_train_1.csv", index=False)

# Create an input space
# This is useful for visualising model perfromance
input_space = np.linspace(0, 6, 100)

# Save as csv
input_space_df = pd.DataFrame(input_space, columns=["x"])
input_space_df.to_csv("data_local/input_space.csv", index=False)

# Define an empty dictionary to hold resource IDs (for convenience)
resource_ids: dict[str,str] = dict()
resource_names = ["x_train_1", "y_train_1", "input_space"]

# Delete existing resources with the same names to avoid conflicts (optional, but useful during development)
for resource_name in resource_names:
    try:
        existing_resource_id = client.resources.get_resource_id_by_name(

                                    resource_type="dataset"
                                    , project_id=project_id
                                    , name=resource_name

                                )

        client.resources.delete_resource(
            
            project_id=project_id
            , resource_type="dataset"
            , resource_id=existing_resource_id
            
        )

        print(f'Deleted existing resource "{resource_name}" with ID {existing_resource_id}')

    except:
        print(f'No existing resource named "{resource_name}" found. Proceeding to upload.')


# Upload the datasets and store their resource IDs
for resource_name, file_path in zip(

    resource_names
    , ["data_local/x_train_1.csv", "data_local/y_train_1.csv", "data_local/input_space.csv"]

):
    resource_id = client.resources.upload(

        project_id=project_id
        , name=resource_name
        , file_path=file_path
        , resource_type="dataset"
        
    )

    resource_ids[resource_name] = resource_id
    print(f'Uploaded {file_path} as resource "{resource_name}"')

### 2. Automated Design of Experiments and Sampling

In [ ]:
# We start with two datapoints at the extremities of out input space [0, 5]
# The number of iterations we set is essentially the BUDGET FOR FURTHER SAMPLES
# Select the number of activate learning loop iteration desired

iterations = 10

#### Reusable workflow(s)

Reusable workflows have been constructed using the **Uncertainty Engine SDK** in `utils.py` to perform the following functions:

---

#### **1. Sample Visualisation**
- Plot the samples available in the current iteration  
- Superimpose these samples on the *ground truth*

---

#### **2. Model Training (Gaussian Process)**
Train a **Gaussian Process (GP)** — a probabilistic ML model — using the available samples for:

- *Uncertainty Quantification (UQ)*
- *Recommendation of the next sample* (**active learning**)

---

#### **3. Model Visualisation (Uncertainty Plot)**
Visualise the trained model to show:

- **Predicted (Posterior) Mean**  
  *→ i.e., the actual model - mapping input(s) → output(s)*

- **Credible Intervals (67% & 99%)**  
  *→ Derived from the posterior standard deviation*  
  *→ Represents uncertainty in the model*

---

#### **4. Next Sample Recommendation**
- Recommend the next input point to sample based on the chosen *acquisition function*  

> **Note:** The default acquisition function is the *posterior standard deviation*

---

#### **5. Update Dataset (Simulation Step)**
- Simulate a new sample at the recommended input value  
- Append this sample to the dataset  

> **Note:**  
> To track changes across iterations we store the training data from each iteration:  
> - `x_train_{iteration}`  
> - `y_train_{iteration}`

---

### 🔁 **Iteration Loop**
Repeat the above steps for the specified number of iterations.

In [ ]:
for iteration in range(1, iterations+1):

    print('-------'*25)
    if iteration == 1:
        print( 'Iteration: ' + str(iteration))
    else:
        print( 'Iteration: ' + str(iteration) + ' (new sample recommended at ' + str(np.round(recommended_dpa, 3)) + ' dpa)')
    print('-------'*25)

    # 1. Sample Visualisation

    training_dpa = get_dataset(
                        client=client
                        , resource_name="x_train_" + str(iteration)
                        , project_name=PROJECT_NAME
                    )
    training_fs = get_dataset(
                        client=client
                        , resource_name="y_train_" + str(iteration)
                        , project_name=PROJECT_NAME
                    )

    plot_basic(training_dpa, training_fs)

    project_id = client.projects.get_project_id_by_name(PROJECT_NAME)

    # 2. Model Training (GP)

    train_model(

        client=client
        , project_id=project_id
        , iteration=iteration

    )

    # 3. Model Visualisation (UQ)

    create_visualise_dataset(

        client=client 
        , project_id=project_id
        , input_space_name="input_space"
        , iteration=iteration
        
    )

    training_dpa = get_dataset(
                        client=client
                        , resource_name="x_train_" + str(iteration)
                        , project_name=PROJECT_NAME
                    )
    training_fs = get_dataset(
                        client=client
                        , resource_name="y_train_" + str(iteration)
                        , project_name=PROJECT_NAME
                    )
    visualise_means = get_dataset(
                        client=client
                        , resource_name="mean_" + str(iteration)
                        , project_name=PROJECT_NAME
                    )
    visualise_stds = get_dataset(
                        client=client
                        , resource_name="std_" + str(iteration)
                        , project_name=PROJECT_NAME
                    )
    visualise_inputs = get_dataset(
                        client=client
                        , resource_name="input_space" 
                        , project_name=PROJECT_NAME
                    )
    # Option to keep or remove the ground truth as a reference
    with_ground_truth = True

    plot_model_uncertainty(

        training_dpa=training_dpa
        , training_fs=training_fs
        , visualise_means=visualise_means
        , visualise_stds=visualise_stds
        , visualise_inputs=visualise_inputs
        , with_ground_truth=with_ground_truth
        , iteration=iteration

    )

    # 4. Next Sample Recommendation

    recommended_dpa = recommend_new_sample(
    
        client=client
        , project_id=project_id
        , iteration=iteration
        , acquisition_function="PosteriorStandardDeviation"
        
    )

    print('Recommended dpa: ' + str(np.round(recommended_dpa, 3)))

    # Optional to add noise contribution to simulation
    new_fs = simulate(recommended_dpa, noise=False)

    print('-------'*25)
    print('New fs for ' + str(np.round(recommended_dpa, 3)) + ' dpa: ' + str(np.round(new_fs, 2)) + ' MPa')

    # 5. Update Dataset

    new_iteration = iteration + 1

    add_new_sample(
        
        client=client
        , new_dpa=recommended_dpa
        , new_fs=new_fs
        , iteration=new_iteration
        , project_name=PROJECT_NAME

    )

In [ ]:
# Example outputs based on the above code and functions in utils.py:
# Process will vary depending on the chosen 'ground truth' and noise intensity.

from IPython.display import Image, display

# Example 1
display(Image(filename="example.gif"))

# Example 2
display(Image(filename="example2.gif"))